# AlphaFold CPU Benchmark (free Colab)

Runs the **same script** used for the TPU and GPU runs, unmodified.

**Before running:** `Runtime > Change runtime type > CPU` (no accelerator).

## 1. Confirm no accelerator is attached

In [1]:
!nvidia-smi || print("No GPU visible -- correct, this is the CPU baseline run.")

/bin/bash: -c: line 1: syntax error near unexpected token `"No GPU visible -- correct, this is the CPU baseline run."'
/bin/bash: -c: line 1: `nvidia-smi || print("No GPU visible -- correct, this is the CPU baseline run.")'


## 2. Install dependencies (plain CPU jax; Colab's preinstalled TensorFlow is left untouched)

In [2]:
!pip install -q -U jax
!pip install -q dm-haiku ml_collections absl-py biopython numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 91.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.4 MB/s eta 0:00:00


## 3. Clone AlphaFold

In [3]:
!git clone --depth 1 https://github.com/google-deepmind/alphafold.git /content/alphafold

Cloning into '/content/alphafold'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 141 (delta 29), reused 60 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 15.94 MiB | 27.29 MiB/s, done.
Resolving deltas: 100% (29/29), done.


## 4. Write the benchmark script

In [4]:
script_content = r'''"""CPU/GPU/TPU backend benchmark for AlphaFold's JAX/Haiku model.

Same script runs unmodified on CPU, GPU, or TPU -- JAX picks up whatever
backend is installed/visible. Produces a JSON results file with timings
(init/compile, first predict, steady-state) plus a JAX/XLA profiler trace,
so one run doubles as:
  - a data point for the course's required CPU vs GPU vs TPU comparison
    (Slide 4), and
  - the profiling artifact for the Measurements slide (Slide 3).

Goal: answer ONE yes/no question as fast as possible on a new backend --
  "Does AlphaFold's core JAX/Haiku forward pass compile and execute here?"

Deliberately avoids two heavy dependencies that are NOT needed to answer
that question:
  1. jackhmmer/hhblits/hhsearch genetic-database search (MSA) -- we build a
     trivial single-sequence "MSA" by hand using the repo's own
     `pipeline.make_msa_features` helper. Swap in a real ColabFold MSA later;
     the plumbing is identical either way.
  2. Downloaded trained parameters (~350MB/model from GCS) -- RunModel can
     randomly initialize params via Haiku's own init, which exercises the
     exact same JIT-compiled graph shapes and accelerator ops as a real
     forward pass. Structure quality is meaningless here; wall-clock and
     "did XLA compile" are what we care about.

Usage:
    python3 spike_tpu_forward_pass.py --run_tag=tpu-v5e
    python3 spike_tpu_forward_pass.py --run_tag=cpu
    python3 spike_tpu_forward_pass.py --run_tag=gpu-t4

On TPU, install jax[tpu] first so JAX grabs the TPU backend instead of CPU:
    pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
On GPU (e.g. Colab), install jax[cuda12] instead.
"""

import json
import os
import platform
import time

from absl import app
from absl import flags
from absl import logging
import jax

# --- Compatibility shim -----------------------------------------------
# AlphaFold's source (a few-years-old repo) calls jnp.clip(x, a_min=..,
# a_max=..) -- current JAX renamed those kwargs to min=/max= and removed
# the old ones. Patch jnp.clip to accept both, so we don't have to hand-edit
# the cloned AlphaFold source. Must happen BEFORE `from alphafold... import`.
import jax.numpy as jnp
_original_clip = jnp.clip
def _clip_compat(*args, **kwargs):
  if "a_min" in kwargs:
    kwargs["min"] = kwargs.pop("a_min")
  if "a_max" in kwargs:
    kwargs["max"] = kwargs.pop("a_max")
  return _original_clip(*args, **kwargs)
jnp.clip = _clip_compat
# ------------------------------------------------------------------------

from alphafold.data import parsers
from alphafold.data import pipeline
from alphafold.model import config as af_config
from alphafold.model import features as af_features
from alphafold.model import model as af_model

FLAGS = flags.FLAGS
flags.DEFINE_string(
    "run_tag", None,
    "Label for this run, e.g. 'cpu', 'gpu-t4', 'tpu-v5e'. Used to name the "
    "output JSON and the profiler trace dir, and to tag the result for the "
    "CPU/GPU/TPU comparison. Defaults to the detected jax backend if unset.")
flags.DEFINE_string(
    "results_dir", "results",
    "Where to write the per-run JSON result and the profiler trace.")

# A short, made-up ~120-residue sequence. Doesn't need to be biologically
# real for a compile/execution benchmark -- we're testing the accelerator
# pipeline, not folding a real protein.
TOY_SEQUENCE = (
    "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKV"
    "KALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWD"
)
NUM_RES = len(TOY_SEQUENCE)


def build_minimal_features():
  """Builds a model-ready feature dict without any external search tools."""
  logging.info("Building sequence features for a %d-residue toy protein...", NUM_RES)
  seq_features = pipeline.make_sequence_features(
      sequence=TOY_SEQUENCE, description="spike_test", num_res=NUM_RES
  )
  # Trivial single-sequence MSA: just the query itself, no alignment depth.
  # Swap this block for a real ColabFold-fetched A3M once plumbing is proven.
  toy_msa = parsers.Msa(
      sequences=[TOY_SEQUENCE],
      deletion_matrix=[[0] * NUM_RES],
      descriptions=["query"],
  )
  msa_features = pipeline.make_msa_features(msas=[toy_msa])
  return {**seq_features, **msa_features}


def main(_):
  backend = jax.default_backend()
  run_tag = FLAGS.run_tag or backend
  os.makedirs(FLAGS.results_dir, exist_ok=True)
  trace_dir = os.path.join(FLAGS.results_dir, f"trace_{run_tag}")

  logging.info("JAX backend: %s", backend)
  logging.info("JAX devices: %s", jax.devices())
  logging.info("Run tag: %s (results -> %s)", run_tag, FLAGS.results_dir)

  cfg = af_config.model_config("model_3")  # use_templates=False by default
  cfg.model.num_recycle = 0
  cfg.data.common.num_recycle = 0
  cfg.data.eval.num_ensemble = 1

  raw_features = build_minimal_features()

  logging.info("Running TF feature-processing pipeline...")
  processed_features = af_features.np_example_to_features(
      np_example=raw_features, config=cfg, random_seed=0
  )

  logging.info("Constructing RunModel (no trained params -> random init)...")
  runner = af_model.RunModel(cfg, params=None)

  logging.info("Calling init_params (first JIT trace)...")
  t0 = time.time()
  runner.init_params(processed_features, random_seed=0)
  t_init = time.time() - t0
  logging.info("init_params done in %.1fs", t_init)

  # JAX/XLA profiler trace -- the "Telemetry" artifact the rubric asks for
  # (Slide 3: Measurements). View later with:
  #   pip install tensorboard-plugin-profile
  #   tensorboard --logdir=<trace_dir>
  logging.info("First predict() call -- XLA COMPILE + run (profiler trace on)...")
  t0 = time.time()
  with jax.profiler.trace(trace_dir):
    result = runner.predict(processed_features, random_seed=0)
    jax.block_until_ready(result)
  t_compile_and_run = time.time() - t0
  logging.info("First predict() done in %.1fs (compile + execute)", t_compile_and_run)

  logging.info("Second predict() call -- should be compiled already (steady-state)...")
  t0 = time.time()
  result2 = runner.predict(processed_features, random_seed=0)
  jax.block_until_ready(result2)
  t_steady_state = time.time() - t0
  logging.info("Second predict() done in %.2fs (steady-state, no compile)", t_steady_state)

  summary = {
      "run_tag": run_tag,
      "backend": backend,
      "devices": [str(d) for d in jax.devices()],
      "num_devices": jax.device_count(),
      "host_processor": platform.processor(),
      "host_machine": platform.machine(),
      "num_residues": NUM_RES,
      "init_params_seconds": round(t_init, 2),
      "first_predict_compile_and_run_seconds": round(t_compile_and_run, 2),
      "second_predict_steady_state_seconds": round(t_steady_state, 3),
      "output_final_atom_positions_shape": list(
          result["structure_module"]["final_atom_positions"].shape
      ),
  }
  out_path = os.path.join(FLAGS.results_dir, f"result_{run_tag}.json")
  with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

  print("\n" + "=" * 60)
  print("SPIKE RESULT")
  print("=" * 60)
  for k, v in summary.items():
    print(f"  {k:38s}: {v}")
  print(f"  {'results JSON':38s}: {out_path}")
  print(f"  {'profiler trace':38s}: {trace_dir}")
  print("=" * 60)
  print("If you got here with no exceptions: YES, AlphaFold's JAX/Haiku")
  print("model compiles and runs on this backend.")
  print("=" * 60)


if __name__ == "__main__":
  app.run(main)
'''
with open('/content/alphafold/spike_tpu_forward_pass.py', 'w') as f:
    f.write(script_content)
import os
print('Wrote', len(script_content), 'bytes')
print('Exists:', os.path.exists('/content/alphafold/spike_tpu_forward_pass.py'))

Wrote 7449 bytes
Exists: True


## 5. Run it -- tagged `cpu-colab`

In [5]:
%cd /content/alphafold
!python3 spike_tpu_forward_pass.py --run_tag=cpu-colab

/content/alphafold
2026-08-08 05:19:35.527417: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0808 05:19:35.568934 131975473233920 xla_bridge.py:836] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0808 05:19:35.569798 131975473233920 spike_tpu_forward_pass.py:111] JAX backend: cpu
I0808 05:19:35.570129 131975473233920 spike_tpu_forward_pass.py:112] JAX devices: [CpuDevice(id=0)]
I0808 05:19:35.570276 131975473233920 spike_tpu_forward_pass.py:113] Run tag: cpu-colab (results -> results)
I0808 05:19:35.572563 131975473233920 spike_tpu_forward_pass.py:90] Building sequence features for a 118-residue toy protein...
I0808 05:19:35.572868 131975473233920 spike_tpu_forward_pass.py:122] Running TF feature-processing pipeline...
I0000 00:00:1786166375.971492    1315 mlir_graph_optimization_pass

## 6. Print the JSON result

In [6]:
!cat /content/alphafold/results/result_cpu-colab.json

{
  "run_tag": "cpu-colab",
  "backend": "cpu",
  "devices": [
    "cpu:0"
  ],
  "num_devices": 1,
  "host_processor": "x86_64",
  "host_machine": "x86_64",
  "num_residues": 118,
  "init_params_seconds": 41.99,
  "first_predict_compile_and_run_seconds": 271.98,
  "second_predict_steady_state_seconds": 212.113,
  "output_final_atom_positions_shape": [
    118,
    37,
    3
  ]
}

## 7. Download the result file

In [7]:
from google.colab import files
files.download('/content/alphafold/results/result_cpu-colab.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>